# Bab 1: Pengenalan Pola sebagai Inferensi

Notebook pendamping buku *Pengenalan Pola dan Machine Learning: Pendekatan Probabilistik dengan Python*.

Jalankan sel dari atas ke bawah (**Runtime → Run all** di Colab). Sel pertama menyiapkan folder kode dan modul data sintetis; setiap sel berikutnya adalah satu listing dari buku, dengan nomor penanda `# (1)`, `# (2)`, … yang dijelaskan di bawah listing pada buku. Angka keluaran bisa sedikit berbeda dari buku bila versi pustakanya berbeda (lihat Lampiran A).

In [ ]:
# Menyiapkan folder kode/ dan modul pendukung bab ini
import os, pathlib
BERKAS = {
"bab01/map_ridge.py": "import numpy as np\n\nrng = np.random.default_rng(7)\nf = lambda x: 200 + 150 * np.cos(2 * np.pi * x)\nx_latih = rng.uniform(0, 1, 12); t_latih = f(x_latih) + rng.normal(0, 40, 12)\nx_uji = rng.uniform(0, 1, 1000); t_uji = f(x_uji) + rng.normal(0, 40, 1000)\n\nM = 9\nPhi = lambda x: np.vander(x, M + 1, increasing=True)  # 1, x, ..., x^M\nbeta = 1 / 40**2                                      # presisi derau  #<1>\n\nprint(\" alpha      lambda=alpha/beta   RMSE uji (mm)   |w| maks\")\nfor alpha in [1e-10, 1e-8, 1e-6, 1e-4, 1e-2]:          # presisi prior  #<2>\n    A = alpha * np.eye(M + 1) + beta * Phi(x_latih).T @ Phi(x_latih)\n    w_map = np.linalg.solve(A, beta * Phi(x_latih).T @ t_latih)  #<3>\n    rmse = np.sqrt(np.mean((Phi(x_uji) @ w_map - t_uji) ** 2))\n    print(f\"{alpha:6.0e}   {alpha / beta:17.1e}   {rmse:13.1f}   {np.abs(w_map).max():9.0f}\")\n",
"bab01/mle_derau.py": "import numpy as np\n\nrng = np.random.default_rng(7)\nf = lambda x: 200 + 150 * np.cos(2 * np.pi * x)\nM = 5                                            # cukup lentur untuk pola monsun\nfor N in [12, 50, 500]:\n    var_ml = []\n    for _ in range(2000):                        # ulangi eksperimen 2000 kali  #<1>\n        x = rng.uniform(0, 1, N)\n        t = f(x) + rng.normal(0, 40, N)\n        w = np.polyfit(x, t, M)                  # w_ML = solusi kuadrat terkecil\n        var_ml.append(np.mean((np.polyval(w, x) - t) ** 2))  # sigma^2_ML  #<2>\n    v = np.mean(var_ml)\n    v_kor = v * N / (N - (M + 1))                # koreksi derajat bebas  #<3>\n    print(f\"N={N:4d}   sigma_ML = {np.sqrt(v):4.1f} mm   \"\n          f\"terkoreksi = {np.sqrt(v_kor):4.1f} mm   (sebenarnya 40)\")\n",
"bab01/polinomial_hujan.py": "import numpy as np\n\nrng = np.random.default_rng(7)\ndef curah_hujan(x):                        # pola monsun \"sebenarnya\" (mm/bulan)\n    return 200 + 150 * np.cos(2 * np.pi * x)  #<1>\n\ndef buat_data(n):\n    x = rng.uniform(0, 1, n)               # waktu dalam setahun, 0 = 1 Januari\n    t = curah_hujan(x) + rng.normal(0, 40, n)  # derau cuaca, sigma = 40 mm\n    return x, t\n\nx_latih, t_latih = buat_data(12)           # hanya 12 pengamatan  #<2>\nx_uji, t_uji = buat_data(1000)\n\ndef rmse(w, x, t):\n    return np.sqrt(np.mean((np.polyval(w, x) - t) ** 2))\n\nprint(\"orde M   RMSE latih (mm)   RMSE uji (mm)\")\nfor M in [0, 1, 3, 9]:\n    w = np.polyfit(x_latih, t_latih, M)    # kuadrat terkecil  #<3>\n    print(f\"{M:6d}   {rmse(w, x_latih, t_latih):15.1f}   {rmse(w, x_uji, t_uji):13.1f}\")\n",
"bab01/prediksi_bayes.py": "import numpy as np\n\nrng = np.random.default_rng(3)\nf = lambda x: 200 + 150 * np.cos(2 * np.pi * x)\nx = rng.uniform(0, 1, 40)\nx = x[(x < 5/12) | (x > 8/12)][:15]          # alat rusak Juni-Agustus  #<1>\nt = f(x) + rng.normal(0, 40, x.size)\n\nM, alpha, beta = 9, 1e-8, 1 / 40**2\nPhi = lambda x: np.vander(np.atleast_1d(x), M + 1, increasing=True)\nS = np.linalg.inv(alpha * np.eye(M + 1) + beta * Phi(x).T @ Phi(x))  #<2>\nm = beta * S @ Phi(x).T @ t\n\ndef prediksi(x_baru):                        # distribusi prediktif  #<3>\n    phi = Phi(x_baru)\n    rerata = phi @ m\n    ragam_model = np.sum(phi @ S * phi, axis=1)  # ketidakpastian tentang w\n    return rerata, np.sqrt(ragam_model), np.sqrt(1 / beta + ragam_model)\n\nprint(f\"titik latih: {x.size}, tanpa data antara Juni dan Agustus\")\nprint(\"bulan     rerata  s_model  s_total  sebenarnya   (semua dalam mm)\")\nfor nama, xb in [(\"Januari\", 0.5/12), (\"April\", 3.5/12),\n                 (\"Juli\", 6.5/12), (\"Oktober\", 9.5/12)]:\n    r, s_m, s_t = prediksi(xb)\n    print(f\"{nama:8s} {r[0]:7.0f} {s_m[0]:8.0f} {s_t[0]:8.0f} {f(xb):11.0f}\")\n"
}
akar = pathlib.Path('pola-ml/kode').resolve()
for nama, isi in BERKAS.items():
    p = akar / nama; p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(isi, encoding='utf-8')
os.chdir(akar / 'bab01')
print('folder kerja:', os.getcwd())

## Listing 1.1: Mencocokkan polinom pada data curah hujan sintetis

Berkas: `kode/bab01/polinomial_hujan.py`

In [ ]:
import numpy as np

rng = np.random.default_rng(7)
def curah_hujan(x):                        # pola monsun "sebenarnya" (mm/bulan)
    return 200 + 150 * np.cos(2 * np.pi * x)  # (1)

def buat_data(n):
    x = rng.uniform(0, 1, n)               # waktu dalam setahun, 0 = 1 Januari
    t = curah_hujan(x) + rng.normal(0, 40, n)  # derau cuaca, sigma = 40 mm
    return x, t

x_latih, t_latih = buat_data(12)           # hanya 12 pengamatan  # (2)
x_uji, t_uji = buat_data(1000)

def rmse(w, x, t):
    return np.sqrt(np.mean((np.polyval(w, x) - t) ** 2))

print("orde M   RMSE latih (mm)   RMSE uji (mm)")
for M in [0, 1, 3, 9]:
    w = np.polyfit(x_latih, t_latih, M)    # kuadrat terkecil  # (3)
    print(f"{M:6d}   {rmse(w, x_latih, t_latih):15.1f}   {rmse(w, x_uji, t_uji):13.1f}")

## Listing 1.2: Bias taksiran maximum likelihood untuk varians derau

Berkas: `kode/bab01/mle_derau.py`

In [ ]:
import numpy as np

rng = np.random.default_rng(7)
f = lambda x: 200 + 150 * np.cos(2 * np.pi * x)
M = 5                                            # cukup lentur untuk pola monsun
for N in [12, 50, 500]:
    var_ml = []
    for _ in range(2000):                        # ulangi eksperimen 2000 kali  # (1)
        x = rng.uniform(0, 1, N)
        t = f(x) + rng.normal(0, 40, N)
        w = np.polyfit(x, t, M)                  # w_ML = solusi kuadrat terkecil
        var_ml.append(np.mean((np.polyval(w, x) - t) ** 2))  # sigma^2_ML  # (2)
    v = np.mean(var_ml)
    v_kor = v * N / (N - (M + 1))                # koreksi derajat bebas  # (3)
    print(f"N={N:4d}   sigma_ML = {np.sqrt(v):4.1f} mm   "
          f"terkoreksi = {np.sqrt(v_kor):4.1f} mm   (sebenarnya 40)")

## Listing 1.3: Estimasi MAP: prior Gaussian sebagai regularisasi

Berkas: `kode/bab01/map_ridge.py`

In [ ]:
import numpy as np

rng = np.random.default_rng(7)
f = lambda x: 200 + 150 * np.cos(2 * np.pi * x)
x_latih = rng.uniform(0, 1, 12); t_latih = f(x_latih) + rng.normal(0, 40, 12)
x_uji = rng.uniform(0, 1, 1000); t_uji = f(x_uji) + rng.normal(0, 40, 1000)

M = 9
Phi = lambda x: np.vander(x, M + 1, increasing=True)  # 1, x, ..., x^M
beta = 1 / 40**2                                      # presisi derau  # (1)

print(" alpha      lambda=alpha/beta   RMSE uji (mm)   |w| maks")
for alpha in [1e-10, 1e-8, 1e-6, 1e-4, 1e-2]:          # presisi prior  # (2)
    A = alpha * np.eye(M + 1) + beta * Phi(x_latih).T @ Phi(x_latih)
    w_map = np.linalg.solve(A, beta * Phi(x_latih).T @ t_latih)  # (3)
    rmse = np.sqrt(np.mean((Phi(x_uji) @ w_map - t_uji) ** 2))
    print(f"{alpha:6.0e}   {alpha / beta:17.1e}   {rmse:13.1f}   {np.abs(w_map).max():9.0f}")

## Listing 1.4: Distribusi prediktif Bayesian untuk regresi polinom

Berkas: `kode/bab01/prediksi_bayes.py`

In [ ]:
import numpy as np

rng = np.random.default_rng(3)
f = lambda x: 200 + 150 * np.cos(2 * np.pi * x)
x = rng.uniform(0, 1, 40)
x = x[(x < 5/12) | (x > 8/12)][:15]          # alat rusak Juni-Agustus  # (1)
t = f(x) + rng.normal(0, 40, x.size)

M, alpha, beta = 9, 1e-8, 1 / 40**2
Phi = lambda x: np.vander(np.atleast_1d(x), M + 1, increasing=True)
S = np.linalg.inv(alpha * np.eye(M + 1) + beta * Phi(x).T @ Phi(x))  # (2)
m = beta * S @ Phi(x).T @ t

def prediksi(x_baru):                        # distribusi prediktif  # (3)
    phi = Phi(x_baru)
    rerata = phi @ m
    ragam_model = np.sum(phi @ S * phi, axis=1)  # ketidakpastian tentang w
    return rerata, np.sqrt(ragam_model), np.sqrt(1 / beta + ragam_model)

print(f"titik latih: {x.size}, tanpa data antara Juni dan Agustus")
print("bulan     rerata  s_model  s_total  sebenarnya   (semua dalam mm)")
for nama, xb in [("Januari", 0.5/12), ("April", 3.5/12),
                 ("Juli", 6.5/12), ("Oktober", 9.5/12)]:
    r, s_m, s_t = prediksi(xb)
    print(f"{nama:8s} {r[0]:7.0f} {s_m[0]:8.0f} {s_t[0]:8.0f} {f(xb):11.0f}")